# 只补2026：ER＋V2官方日档修订对照

旧结果与训练参数保留；2020—2025不重跑。两份旧ZIP缺失且官方版本已变，因此新目录里统一重新计算2026基准与入选变体，避免只给失败组换数据。不是原数据精确复现。2026年初各币500U至北京时间11月15日全天，ZERO／零资金费／.02%退出滑点／RAW逐笔。

按1→6。第4格只准备，第5格才计算。无需GPU。未扩大实验网格，不重新训练；不改main、旧账本或部署。

## 1. 环境

In [ ]:
import sys,os,subprocess,shutil,platform
assert sys.version_info >= (3,12), '请使用Python3.12或更高的Colab运行时'
subprocess.run([sys.executable,'-m','pip','-q','install','numpy==2.3.5','pandas==3.0.1','psutil==7.0.0'],check=True)
import psutil
print('Python:',sys.version.split()[0], 'CPU:',os.cpu_count(),platform.machine())
print('RAM available/total GB:',round(psutil.virtual_memory().available/2**30,1),round(psutil.virtual_memory().total/2**30,1))
print('Local disk free GB:',round(shutil.disk_usage('/content').free/2**30,1))
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
else: print('GPU: none. This financial replay uses CPU.')

## 2. 锁定代码
读取私有GitHub仓库需要Colab Secret GITHUB_TOKEN，或隐藏输入，不保存或打印凭据。

In [ ]:
from pathlib import Path
import tempfile,getpass,json
CODE_COMMIT = 'ae8fbb1204a6fcd9c822a42fbb78cc8ec591066a'
REPO = Path('/content/er-v2-2026-revision-code-'+CODE_COMMIT[:7])
if (REPO/'.git').exists():
    assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT, '目录提交不一致；请使用新目录，不覆盖旧任务'
else:
    if REPO.exists() and any(REPO.iterdir()): raise RuntimeError('目标目录非空；换一个新目录')
    try:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub private repository token (hidden): ')
    if not token: raise RuntimeError('缺少私有仓库读取凭据')
    env=os.environ.copy();env['COLAB_GH_TOKEN']=token;env['GIT_TERMINAL_PROMPT']='0'
    with tempfile.TemporaryDirectory() as tmp:
        ask=Path(tmp)/'askpass.py'
        ask.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["COLAB_GH_TOKEN"])\n')
        ask.chmod(0o700);env['GIT_ASKPASS']=str(ask)
        try:
            REPO.mkdir(parents=True,exist_ok=True)
            for cmd in [['git','init','-q'],['git','remote','add','origin','https://github.com/HITGoose/btc-eth-backtest.git'],['git','-c','credential.helper=','fetch','-q','--depth=1','origin',CODE_COMMIT],['git','checkout','-q','--detach','FETCH_HEAD']]:
                subprocess.run(cmd,cwd=REPO,env=env,check=True,capture_output=True)
        finally:
            env.pop('COLAB_GH_TOKEN',None);del token
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==CODE_COMMIT
os.environ['BTC_REPO_ROOT']=str(REPO)
ENTRY=REPO/'outputs/er_v2_2026_revision'
print('Locked implementation:',CODE_COMMIT)
print('Repository:',REPO)

## 3. Drive及独立恢复目录
旧结果目录只读。新结果保存到原共享目录的子目录，因此仍可从原共享链接访问。只恢复本次新版目录的断点，不导入旧2026断点。市场缓存先复制到本地计算。

In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
SOURCE=DRIVE/'btc_eth_er_v2_retests/6a26255'
assert SOURCE.is_dir(), '找不到原Drive结果目录'
DRIVE_DATA=DRIVE/'btc_eth_market_cache'
DRIVE_RESULTS=SOURCE/('2026_OFFICIAL_REVISION_'+CODE_COMMIT[:7])
DATA=Path('/content/btc_eth_market_cache_revision')
RESULTS=Path('/content/er_v2_2026_revision_results_'+CODE_COMMIT[:7])
def restore(source,target,prior_only=False):
    if not source.exists(): return
    for p in source.rglob('*'):
        if not p.is_file() or p.name.endswith(('.copying','.tmp','.part','.restoring')): continue
        if prior_only and (p.suffix not in ['.json','.npz'] or 'smoke' in p.parts): continue
        dest=target/p.relative_to(source)
        if dest.exists() and dest.stat().st_mtime>=p.stat().st_mtime and dest.stat().st_size==p.stat().st_size: continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+'.restoring');shutil.copy2(p,temp);temp.replace(dest)
DATA.mkdir(parents=True,exist_ok=True);RESULTS.mkdir(parents=True,exist_ok=True)
restore(DRIVE_DATA,DATA);restore(DRIVE_RESULTS,RESULTS)
os.environ.update(BTC_REPO_ROOT=str(REPO),BTC_DATA_ROOT=str(DATA),BTC_OUTPUT_ROOT=str(RESULTS),BTC_DRIVE_DATA=str(DRIVE_DATA),BTC_DRIVE_RESULTS=str(DRIVE_RESULTS),PYTHONUNBUFFERED='1')
print('旧结果只读：',SOURCE)
print('本次结果和断点：',DRIVE_RESULTS)
print('尚未启动账户回放。')


## 4. 准备和锁定数据（不启动回测）
核验原四份训练冻结文件SHA；仅复制2020—2025已完成账本。获取两份官方CHECKSUM并冻结新数据清单，下载验证后将确切ZIP另存Drive。其他日期和指标输入仍严格校验；任何异常仍拦截。已在Codex做8小时ER＋V2调用与原分钟权益精确核对，未运行年度矩阵。

In [ ]:
BASE_CMD=[sys.executable,'-u',str(ENTRY/'run.py'),'--source',str(SOURCE)]
subprocess.run(BASE_CMD+['--prepare-only'],cwd=REPO,check=True)
print('准备完成，尚未运行2026账户。下一格才开始。')


## 5. 仅2026正式计算
从2026年初统一回放全部入选策略，原2020—2025不重跑；训练赢家和固定杠杆不改。CPU内存限制最多两币并行。停止后重新运行1→4，再5，本次同指纹/同数据版本的完成账户跳过，其他从本次兼容日断点恢复。不要运行原ER Notebook⑤补这个数据版本。

In [ ]:
import signal
process=subprocess.Popen(BASE_CMD,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
try:
    for line in process.stdout:print(line,end='',flush=True)
    if process.wait()!=0:raise RuntimeError('运行中断或阻断，查看错误；已有结果和断点保留。')
except KeyboardInterrupt:
    try:os.killpg(process.pid,signal.SIGTERM)
    except ProcessLookupError:pass
    process.wait();raise
print('流程结束。查看incomplete.csv确认是否全部完成。')


## 6. 只读汇总
原2020—2025＋本次统一新版2026；不混入旧2026结果。修正报告的方向类型比较和同身份期末持仓归因，不修改旧财务数字。不需要上传大ZIP，结果自动保留原Drive共享目录子目录。

In [ ]:
subprocess.run(BASE_CMD+['--report-only'],cwd=REPO,check=True)
print((RESULTS/'REPORT.md').read_text())
print('结果目录：',DRIVE_RESULTS)
print('把这个子目录路径告诉我即可，我可通过原共享链接读取。')


## 可选：单独下载CSV（不重跑）

In [ ]:
from google.colab import files
FILE='annual.csv' # 或 incomplete.csv、trade_matching.csv、REPORT.md、data_revision.json
p=RESULTS/FILE;assert p.is_file(), '文件尚未生成'
files.download(str(p))
